# T2.5 Analyze the models with respect to different groups


In [41]:
from __future__ import annotations

import re
import math
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from matplotlib.cm import ScalarMappable
from matplotlib.colors import ListedColormap, Normalize
from matplotlib.patches import Patch

from scipy import stats
from pathlib import Path
from collections.abc import Callable

from source.metrics.util import RecommenderDataLoader
from source.metrics.beyond_accuracy import BeyondAccuracyMetrics, jaccard_distance
from source.metrics.ranking_metrics import RankingMetrics

from dataclasses import dataclass
from itertools import groupby, product

In [42]:
# Config
MODELS = list(
    dict.fromkeys(
        [
            "BPR",
            "EASE",
            "FISM",
            "ItemKNN",
            "UserKNN",
            "SLIMElastic",
            "NeuMF",
            "NeuMF",
            "LightGCN",
            "NGCF",
            # "Pop",
            # "Random",
            "TunedAugmentation",
            "TunedCascade",
            "TunedCombination",
            "TunedMetalevel",
            "TunedMixed",
            "TunedSwitching",
            "TunedWeighted",
        ]
    )
)
MODEL_COLORS = dict(zip(MODELS, plt.cm.tab20.colors))  # one colour per model, used in every plot
K = 10
ALPHA = 0.05  # FDR level used when flagging cells

HIDDEN_SPLITS = ("train", "validation")  # never re-recommended
TEST_SPLIT = "test"  # "unassigned" pairs remain candidates

ACTIVITY_LABELS = ["low", "mid-low", "mid-high", "high"]
POPULARITY_LABELS = ["tail", "mid", "head"]
USER_DIMS = ["age_group", "gender", "occupation", "activity"]

In [43]:
def load_evaluation_data(prediction_file: str):
    """
    Helper funtion to load scores, align IDs, and build boolean matrices.
    Returns: (user_id2idx, item_id2idx, predicted_scores, train_mask, test_mask, validation_mask)
    """
    with RecommenderDataLoader(base_dir="../../results") as loader:
        users, items, predicted_scores = loader.load_predictions(npz_filename=prediction_file, npz_dir="tuned_predictions")
        num_users, num_items = predicted_scores.shape

        user_id2idx = {int(raw_id): idx for idx, raw_id in enumerate(users)}
        item_id2idx = {int(raw_id): idx for idx, raw_id in enumerate(items)}

        def build_dense_mask(filename: str) -> np.ndarray:
            mask = np.zeros((num_users, num_items), dtype=bool)
            try:
                pairs = loader.load_split_data_file(filename)
                for row in pairs:
                    if len(row) >= 2:
                        raw_u, raw_i = int(row[0]), int(row[1])
                        if raw_u in user_id2idx and raw_i in item_id2idx:
                            mask[user_id2idx[raw_u], item_id2idx[raw_i]] = True
            except (FileNotFoundError, AssertionError) as e:
                print(f"Warning: {filename} skipped or failed. {e}")
            return mask

        train_mask = build_dense_mask("train.tsv")
        test_mask = build_dense_mask("test.tsv")
        validation_mask = build_dense_mask("valid.tsv")

    return user_id2idx, item_id2idx, predicted_scores, train_mask, test_mask, validation_mask


def build_wide_evaluation_dataframe_v2(
    model_names: list[str],
    loader_func: Callable,
    user_info: dict[int, tuple[int, str, str, str]],
    item_info: dict[int, tuple[str, int, list[str]]],
    ground_truth: dict[tuple[int, int], float],
) -> pd.DataFrame:
    """
    Build one row per (user_id, item_id) pair.

    Columns include:
      - user_id, item_id
      - data_split_type
      - one prediction column per model
      - ground_truth
      - user and item metadata

    Predictions and masks are aligned by raw IDs, not assumed
    to have identical positional ordering across models.
    """
    if not model_names:
        raise ValueError("model_names must contain at least one model.")

    df = None
    base_users = None
    base_items = None
    base_split = None

    for model_name in model_names:
        filename = f"{model_name}.npz"

        user_id2idx, item_id2idx, predicted_scores, train_mask, test_mask, validation_mask = loader_func(filename)

        predicted_scores = np.asarray(predicted_scores)
        train_mask = np.asarray(train_mask, dtype=bool)
        test_mask = np.asarray(test_mask, dtype=bool)
        validation_mask = np.asarray(validation_mask, dtype=bool)

        num_users, num_items = predicted_scores.shape
        expected_shape = (num_users, num_items)
        if train_mask.shape != expected_shape or test_mask.shape != expected_shape or validation_mask.shape != expected_shape:
            raise ValueError(f"{model_name}: prediction and mask shapes do not match.")

        # Recover raw IDs in their actual matrix-index order.
        def ids_in_index_order(id2idx, expected_length, name):
            ids = np.empty(expected_length, dtype=np.int64)
            seen = np.zeros(expected_length, dtype=bool)

            for raw_id, idx in id2idx.items():
                idx = int(idx)
                if idx < 0 or idx >= expected_length or seen[idx]:
                    raise ValueError(f"{model_name}: invalid {name} index mapping.")  # noqa: B023
                ids[idx] = int(raw_id)
                seen[idx] = True

            if not seen.all() or len(id2idx) != expected_length:
                raise ValueError(f"{model_name}: {name} mapping is incomplete.")  # noqa: B023

            return ids

        users = ids_in_index_order(user_id2idx, num_users, "user")
        items = ids_in_index_order(item_id2idx, num_items, "item")

        # A pair must not belong to multiple splits.
        overlap = (train_mask & test_mask) | (train_mask & validation_mask) | (test_mask & validation_mask)
        if overlap.any():
            raise ValueError(f"{model_name}: train/test/validation masks overlap for {int(overlap.sum())} user-item pairs.")

        if df is None:
            # Construct the master grid in the first model's ID order.
            base_users = users.copy()
            base_items = items.copy()

            u_idx, i_idx = np.indices(expected_shape)

            df = pd.DataFrame({"user_id": users[u_idx.ravel()], "item_id": items[i_idx.ravel()]})
            split = np.full(expected_shape, "unassigned", dtype=object)
            split[train_mask] = "train"
            split[test_mask] = "test"
            split[validation_mask] = "validation"
            base_split = split.ravel()
            df["data_split_type"] = base_split

        else:
            # Verify that all models cover the same ID sets.
            assert base_users is not None
            assert base_items is not None
            if set(users) != set(base_users) or set(items) != set(base_items):
                raise ValueError(f"{model_name}: user/item ID sets differ from the first model.")

            # Convert the master grid's ID order to this model's
            # positional order before flattening predictions.
            user_positions = np.array([user_id2idx[int(uid)] for uid in base_users])
            item_positions = np.array([item_id2idx[int(iid)] for iid in base_items])

            predicted_scores = predicted_scores[np.ix_(user_positions, item_positions)]

            current_split = np.full(expected_shape, "unassigned", dtype=object)
            current_split[train_mask] = "train"
            current_split[test_mask] = "test"
            current_split[validation_mask] = "validation"

            current_split = current_split[np.ix_(user_positions, item_positions)].ravel()

            assert base_split is not None
            if not np.array_equal(current_split, base_split):
                raise ValueError(f"{model_name}: split assignments differ from the first model after ID alignment.")

        # Predictions now follow the master dataframe's row order.
        df[model_name] = predicted_scores.ravel().astype(np.float32)

    assert df is not None

    # Map ground truth using the exact (user_id, item_id) pair.
    gt = pd.Series(ground_truth, name="ground_truth", dtype=np.float32)
    pair_index = pd.MultiIndex.from_arrays([df["user_id"], df["item_id"]])
    df["ground_truth"] = gt.reindex(pair_index).to_numpy()

    # Join user metadata.
    user_meta = pd.DataFrame.from_dict(
        user_info,
        orient="index",
        columns=["age", "gender", "occupation", "zip_code"],
    )
    user_meta.index = user_meta.index.astype("int64")
    user_meta.index.name = "user_id"

    # Join item metadata.
    item_meta = pd.DataFrame.from_dict(
        item_info,
        orient="index",
        columns=["title", "year", "genres"],
    )
    item_meta.index = item_meta.index.astype("int64")
    item_meta.index.name = "item_id"

    df = df.merge(user_meta.reset_index(), on="user_id", how="left", validate="many_to_one")
    df = df.merge(item_meta.reset_index(), on="item_id", how="left", validate="many_to_one")

    # Add age groups
    df["age_group"] = pd.cut(
        df["age"],
        bins=[0, 18, 26, 36, 46, 56, 101],
        labels=["<18", "18-25", "26-35", "36-45", "46-55", "56+"],
        right=False,
    )

    for col in ["gender", "occupation", "age_group", "data_split_type"]:
        df[col] = df[col].astype("category")

    # Final integrity checks.
    if df.duplicated(["user_id", "item_id"]).any():
        raise AssertionError("Duplicate user-item pairs found.")

    assert base_users is not None
    assert base_items is not None
    expected_rows = len(base_users) * len(base_items)
    if len(df) != expected_rows:
        raise AssertionError(f"Expected {expected_rows} rows, got {len(df)}.")

    return df

In [44]:
df = build_wide_evaluation_dataframe_v2(
    model_names=MODELS,
    loader_func=load_evaluation_data,
    user_info=RecommenderDataLoader().load_user_info(file_path=Path("..", "..", "data", "ml-100k", "ml-100k.user")),
    item_info=RecommenderDataLoader().load_item_info(file_path=Path("..", "..", "data", "ml-100k", "ml-100k.item")),
    ground_truth=RecommenderDataLoader().load_interaction_ratings(file_path=Path("..", "..", "data", "ml-100k", "ml-100k.inter")),
)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1586126 entries, 0 to 1586125
Data columns (total 28 columns):
 #   Column             Non-Null Count    Dtype   
---  ------             --------------    -----   
 0   user_id            1586126 non-null  int64   
 1   item_id            1586126 non-null  int64   
 2   data_split_type    1586126 non-null  category
 3   BPR                1586126 non-null  float32 
 4   EASE               1586126 non-null  float32 
 5   FISM               1586126 non-null  float32 
 6   ItemKNN            1586126 non-null  float32 
 7   UserKNN            1586126 non-null  float32 
 8   SLIMElastic        1586126 non-null  float32 
 9   NeuMF              1586126 non-null  float32 
 10  LightGCN           1586126 non-null  float32 
 11  NGCF               1586126 non-null  float32 
 12  TunedAugmentation  1586126 non-null  float32 
 13  TunedCascade       1586126 non-null  float32 
 14  TunedCombination   1586126 non-null  float32 
 15  TunedMetalevel 

In [45]:
display(df.groupby("user_id").size().describe())

count     943.0
mean     1682.0
std         0.0
min      1682.0
25%      1682.0
50%      1682.0
75%      1682.0
max      1682.0
dtype: float64

In [46]:
# dimension -> {group label -> boolean mask over matrix rows (users) or columns (items)}
GroupMasks = dict[str, dict[str, np.ndarray]]


@dataclass(frozen=True)
class EvaluationData:
    scores: dict[str, np.ndarray]  # model -> (n_users, n_items)
    hidden: np.ndarray  # (n_users, n_items) bool
    relevant: np.ndarray  # (n_users, n_items) bool
    users: pd.DataFrame  # attributes, row-aligned with the matrices
    items: pd.DataFrame  # attributes, column-aligned with the matrices


def build_evaluation_data(df: pd.DataFrame, models: list[str]) -> EvaluationData:
    """Turn the long (user, item) grid dataframe into aligned matrices."""
    df = df.sort_values(["user_id", "item_id"]).reset_index(drop=True)
    user_ids = np.sort(df["user_id"].unique())
    item_ids = np.sort(df["item_id"].unique())
    shape = (len(user_ids), len(item_ids))
    if len(df) != shape[0] * shape[1]:
        raise ValueError("df is not the full user x item grid.")

    split = df["data_split_type"].astype(str).to_numpy().reshape(shape)
    return EvaluationData(
        scores={m: df[m].to_numpy().reshape(shape) for m in models},
        hidden=np.isin(split, HIDDEN_SPLITS),
        relevant=split == TEST_SPLIT,
        users=df.drop_duplicates("user_id").set_index("user_id").loc[user_ids].copy(),
        items=df.drop_duplicates("item_id").set_index("item_id").loc[item_ids].copy(),
    )


def quartile_labels(values: np.ndarray, labels: list[str]) -> pd.Categorical:
    """Equal-sized buckets; ranking first so ties cannot break qcut."""
    ranks = pd.Series(values).rank(method="first").to_numpy()
    return pd.qcut(ranks, q=len(labels), labels=labels)


def masks_by_category(series: pd.Series) -> dict[str, np.ndarray]:
    """One boolean mask per group, in category order (or sorted order)."""
    if isinstance(series.dtype, pd.CategoricalDtype):
        labels = series.cat.categories
    else:
        labels = sorted(series.dropna().unique())
    values = series.to_numpy()
    return {str(label): values == label for label in labels}


def make_user_groups(data: EvaluationData) -> GroupMasks:
    users = data.users.copy()
    users["activity"] = quartile_labels(data.hidden.sum(axis=1), ACTIVITY_LABELS)
    return {dim: masks_by_category(users[dim]) for dim in USER_DIMS}


def make_item_groups(data: EvaluationData) -> GroupMasks:
    items = data.items.copy()  # `genres` must be a list of strings per item
    items["decade"] = (items["year"] // 10 * 10).astype(str) + "s"
    items["popularity_bucket"] = quartile_labels(data.hidden.sum(axis=0), POPULARITY_LABELS)

    all_genres = sorted({genre for genres in items["genres"] for genre in genres})
    genre_masks = {g: items["genres"].apply(lambda genres, g=g: g in genres).to_numpy() for g in all_genres}

    return {
        "decade": masks_by_category(items["decade"]),
        "popularity_bucket": masks_by_category(items["popularity_bucket"]),
        "genre": genre_masks,
    }


def evaluate(
    data: EvaluationData,
    model: str,
    *,
    user_mask: np.ndarray | None = None,
    item_mask: np.ndarray | None = None,
    distance_matrix: np.ndarray | None = None,
    k: int = K,
) -> dict[str, float]:
    """
    Metrics for one model on a user and/or item group.

    Item groups only restrict which test items count as relevant; the ranking
    still runs over the full catalog. Beyond-accuracy metrics are computed
    only for user groups, and only if a distance matrix is passed.
    """
    scores, hidden, relevant = data.scores[model], data.hidden, data.relevant
    if item_mask is not None:
        relevant = relevant & item_mask[None, :]
    if user_mask is not None:
        scores, hidden, relevant = scores[user_mask], hidden[user_mask], relevant[user_mask]
    if not relevant.any():
        return {}

    metrics = RankingMetrics.calculate_all_metrics(scores, hidden, relevant, k)
    if distance_matrix is not None and item_mask is None:
        metrics |= BeyondAccuracyMetrics.calculate_all_metrics(scores, hidden, relevant, distance_matrix, k)
    return metrics


def user_group_table(data: EvaluationData, user_groups: GroupMasks, distance_matrix: np.ndarray, k: int = K) -> pd.DataFrame:
    rows = [
        {
            "model": model,
            "dim": dim,
            "group": group,
            "n_users": int(mask.sum()),
            **evaluate(data, model, user_mask=mask, distance_matrix=distance_matrix, k=k),
        }
        for model in data.scores
        for dim, groups in user_groups.items()
        for group, mask in groups.items()
    ]
    return pd.DataFrame(rows)


def item_group_table(data: EvaluationData, item_groups: GroupMasks, k: int = K) -> pd.DataFrame:
    rows = [
        {"model": model, "dim": dim, "group": group, "n_test": int((data.relevant & mask).sum()), **evaluate(data, model, item_mask=mask, k=k)}
        for model in data.scores
        for dim, groups in item_groups.items()
        for group, mask in groups.items()
    ]
    return pd.DataFrame(rows)

In [47]:
def per_user_metric(data: EvaluationData, model: str, metric: str, item_mask: np.ndarray, k: int = K) -> np.ndarray:
    relevant = data.relevant & item_mask[None, :]
    return RankingMetrics.calculate_per_user_metrics(data.scores[model], data.hidden, relevant, k)[metric]


def _flatten(groups: GroupMasks) -> list[tuple[str, str, np.ndarray]]:
    return [(dim, group, mask) for dim, by_group in groups.items() for group, mask in by_group.items()]


def screen_cells(
    data: EvaluationData,
    user_groups: GroupMasks,
    item_groups: GroupMasks,
    models: list[str],
    metric: str = f"recall@{K}",
    min_users: int = 20,
    min_test: int = 30,
    k: int = K,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Evaluate every (user group, item group) cell, including the 'all' marginals.

    Returns
    -------
    cells   : one row per (cell, model). `lift` = the model's per-user metric minus
              the mean of the other models, tested with a paired t-test.
    winners : one row per cell: best model, runner-up, and whether the margin is significant.
    """
    if len(models) < 2:
        raise ValueError("Need at least two models.")
    n_users, n_items = data.relevant.shape
    user_groups = {"all": {"all": np.ones(n_users, dtype=bool)}, **user_groups}
    item_groups = {"all": {"all": np.ones(n_items, dtype=bool)}, **item_groups}

    # Rank once per model, then compute per-user metrics once per (item group).
    top_k = {m: RankingMetrics.rank_top_k(data.scores[m], data.hidden, k) for m in models}
    per_user: dict[tuple, np.ndarray] = {}  # (item_dim, item_group) -> (n_models, n_users)
    test_counts: dict[tuple, np.ndarray] = {}
    for idim, ig, imask in _flatten(item_groups):
        relevant = data.relevant & imask[None, :]
        test_counts[idim, ig] = relevant.sum(axis=1)
        per_user[idim, ig] = np.stack(
            [RankingMetrics.calculate_per_user_metrics(data.scores[m], data.hidden, relevant, k, ranked=top_k[m])[metric] for m in models]
        )

    cell_rows, winner_rows = [], []
    for (udim, ug, umask), (idim, ig, _) in product(_flatten(user_groups), _flatten(item_groups)):
        values = per_user[idim, ig][:, umask]
        values = values[:, ~np.isnan(values[0])]  # users with >= 1 test item in this cell
        n, n_test = values.shape[1], int(test_counts[idim, ig][umask].sum())
        if n < min_users or n_test < min_test:
            continue
        cell_key = {"user_dim": udim, "user_group": ug, "item_dim": idim, "item_group": ig, "n_users": n, "n_test": n_test}

        # Lift: each model minus the mean of the others, per user.
        lift = values - (values.sum(axis=0) - values) / (len(models) - 1)
        se = lift.std(axis=1, ddof=1) / np.sqrt(n)
        t = np.divide(lift.mean(axis=1), se, out=np.zeros(len(models)), where=se > 0)
        p = 2 * stats.t.sf(np.abs(t), df=n - 1)
        for j, model in enumerate(models):
            cell_rows.append({**cell_key, "model": model, "score": values[j].mean(), "lift": lift[j].mean(), "se": se[j], "p": p[j]})

        # Winner: best mean score, tested against the runner-up.
        means = values.mean(axis=1)
        best, second = np.argsort(-means)[:2]
        diff = values[best] - values[second]
        w_se = diff.std(ddof=1) / np.sqrt(n)
        w_t = diff.mean() / w_se if w_se > 0 else 0.0
        winner_rows.append(
            {**cell_key, "best": models[best], "runner_up": models[second], "margin": diff.mean(), "p": 2 * stats.t.sf(abs(w_t), df=n - 1)}
        )

    cells, winners = pd.DataFrame(cell_rows), pd.DataFrame(winner_rows)
    cells["lift_q"] = stats.false_discovery_control(cells["p"].to_numpy())  # Benjamini-Hochberg
    winners["margin_q"] = stats.false_discovery_control(winners["p"].to_numpy())
    return cells, winners


def add_interaction(cells: pd.DataFrame) -> pd.DataFrame:
    """Cross cells only, with: lift beyond what the row and column marginals predict (per model)."""
    ua, ia = cells["user_dim"].eq("all"), cells["item_dim"].eq("all")
    user_lift = cells.loc[ia, ["model", "user_dim", "user_group", "lift"]].rename(columns={"lift": "user_lift"})
    item_lift = cells.loc[ua, ["model", "item_dim", "item_group", "lift"]].rename(columns={"lift": "item_lift"})
    overall = cells.loc[ua & ia, ["model", "lift"]].rename(columns={"lift": "overall_lift"})

    cross = (
        cells[~ua & ~ia]
        .merge(user_lift, on=["model", "user_dim", "user_group"])
        .merge(item_lift, on=["model", "item_dim", "item_group"])
        .merge(overall, on="model")
    )
    cross["expected_lift"] = cross["user_lift"] + cross["item_lift"] - cross["overall_lift"]
    cross["interaction"] = cross["lift"] - cross["expected_lift"]

    # Approximate: treats the expected lift as fixed. Good enough for screening.
    z = (cross["interaction"] / cross["se"].replace(0, np.nan)).fillna(0)
    cross["interaction_q"] = stats.false_discovery_control(2 * stats.norm.sf(z.abs()))
    return cross


def rank_dimension_pairs(cross: pd.DataFrame, winners: pd.DataFrame, alpha: float = ALPHA) -> pd.DataFrame:
    """
    Which (user dim x item dim) matrices are worth plotting?

    n_distinct_winners : how many different models win at least one cell (1 = one model wins everywhere, boring)
    share_decisive     : share of cells where the winner is significantly ahead of the runner-up
    share_interacting  : share of (cell, model) pairs that deviate from what the marginals predict
    """
    keys = ["user_dim", "item_dim"]
    interaction = (
        cross.assign(interacting=cross["interaction_q"] < alpha, abs_interaction=cross["interaction"].abs())
        .groupby(keys)
        .agg(share_interacting=("interacting", "mean"), mean_abs_interaction=("abs_interaction", "mean"))
    )
    cross_winners = winners[(winners["user_dim"] != "all") & (winners["item_dim"] != "all")]
    wins = (
        cross_winners.assign(decisive=cross_winners["margin_q"] < alpha)
        .groupby(keys)
        .agg(n_cells=("best", "size"), n_distinct_winners=("best", "nunique"), share_decisive=("decisive", "mean"))
    )
    return wins.join(interaction).sort_values(["n_distinct_winners", "share_interacting"], ascending=False)


def rank_dimension_pairs_v2(
    cross: pd.DataFrame, winners: pd.DataFrame, alpha: float = ALPHA, min_cells: int = 4
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Ranks (user_dim x item_dim) cross-tabulations using two complementary statistical heuristics:
    1. Diversity Ranking: Finds competitive grids where different models win different niches.
    2. Dominance Ranking: Finds landslide grids where one model dominates with a large, decisive margin.

    Parameters
    ----------
    cross : pd.DataFrame
        Interaction table containing cell-level metrics, 'interaction', and 'interaction_q'.
    winners : pd.DataFrame
        Winner table containing 'best', 'margin', and 'margin_q' per cell.
    alpha : float, default=0.05
        Target False Discovery Rate (FDR) threshold for q-values (q < alpha).
    min_cells : int, default=4
        Minimum cell count required to evaluate a dimension pair (filters out tiny grids).

    Returns
    -------
    tuple[pd.DataFrame, pd.DataFrame]
        (diverse_pairs, dominant_pairs) ranked DataFrames.
    """
    keys = ["user_dim", "item_dim"]

    # 1. Filter out global margin rows ('all')
    cross_sub = cross[(cross["user_dim"] != "all") & (cross["item_dim"] != "all")]
    win_sub = winners[(winners["user_dim"] != "all") & (winners["item_dim"] != "all")]

    # 2. Compute interaction stats per dimension pair
    interaction_stats = (
        cross_sub.assign(is_interacting=cross_sub["interaction_q"] < alpha, abs_interaction=cross_sub["interaction"].abs())
        .groupby(keys)
        .agg(share_interacting=("is_interacting", "mean"), mean_abs_interaction=("abs_interaction", "mean"))
    )

    # 3. Compute winner stats per dimension pair
    winner_stats = (
        win_sub.assign(is_decisive=win_sub["margin_q"] < alpha)
        .groupby(keys)
        .agg(
            n_cells=("best", "size"),
            n_distinct_winners=("best", "nunique"),
            share_decisive=("is_decisive", "mean"),
            mean_margin=("margin", "mean"),
            # Top winner frequency
            dominant_share=("best", lambda s: s.value_counts(normalize=True).iloc[0]),
            dominant_model=("best", lambda s: s.value_counts().index[0]),
        )
    )

    # 4. Merge metrics and apply minimum grid size filter
    merged = winner_stats.join(interaction_stats).fillna(0.0)
    merged = merged[merged["n_cells"] >= min_cells].copy()

    # -------------------------------------------------------------------------
    # HEURISTIC 1: Competitive Diversity
    # Prioritizes high winner diversity and strong cell-level interaction effects.
    # -------------------------------------------------------------------------
    diverse_pairs = merged.sort_values(by=["n_distinct_winners", "share_interacting", "mean_margin"], ascending=[False, False, False])

    # -------------------------------------------------------------------------
    # HEURISTIC 2: Model Dominance (Landslides)
    # Prioritizes grids where one model holds a huge share, the victory is
    # statistically decisive, and the mean margin over the runner-up is large.
    # -------------------------------------------------------------------------
    dominant_pairs = merged.sort_values(by=["dominant_share", "share_decisive", "mean_margin"], ascending=[False, False, False])

    return diverse_pairs, dominant_pairs


def notable_cells(cross: pd.DataFrame, by: str = "lift", model: str | None = None, alpha: float = ALPHA, n: int = 15) -> pd.DataFrame:
    """Largest significant cells, ranked by |lift| or |interaction|. `model` filters to one model."""
    frame = cross if model is None else cross[cross["model"] == model]
    return frame[frame[f"{by}_q"] < alpha].sort_values(by, key=np.abs, ascending=False).head(n)

In [48]:
def _star_labels(values: pd.DataFrame, q: pd.DataFrame, alpha: float, show_values: bool) -> pd.DataFrame:
    """Annotation text: the value (optional) plus * where FDR q < alpha. Empty for missing cells."""
    stars = pd.DataFrame(np.where(q < alpha, "*", ""), index=values.index, columns=values.columns)
    text = values.round(2).astype(str) + stars if show_values else stars
    return text.where(values.notna(), "")


def plot_model_profile(
    cells: pd.DataFrame,
    models: list[str],
    side: str = "user",  # "user" -> models x user groups, "item" -> models x item groups
    value: str = "lift",  # "lift" (relative to other models) or "score" (absolute metric)
    metric: str = f"recall@{K}",
    alpha: float = ALPHA,
    output_dir: str | Path = Path("..", "..", "figures"),
) -> Path:
    """Models x groups heatmap for one side: where is each model strong or weak?"""
    dim_col, group_col, other_dim = ("user_dim", "user_group", "item_dim") if side == "user" else ("item_dim", "item_group", "user_dim")
    marginals = cells[(cells[other_dim] == "all") & (cells[dim_col] != "all")]
    keys = list(marginals[[dim_col, group_col]].drop_duplicates().itertuples(index=False, name=None))

    def pivot(column: str) -> pd.DataFrame:
        return marginals.pivot(index="model", columns=[dim_col, group_col], values=column).reindex(
            index=models, columns=pd.MultiIndex.from_tuples(keys)
        )

    values, q = pivot(value), pivot("lift_q")
    fig, ax = plt.subplots(figsize=(max(12, 0.6 * len(keys)), 0.6 * len(models) + 3), constrained_layout=True)
    if value == "lift":
        limit = float(np.nanquantile(np.abs(values.to_numpy(dtype=float)), 0.98))
        limit = limit or 1.0
        kwargs = {"cmap": "RdBu", "vmin": -limit, "vmax": limit}
        cbar_label = f"{metric}: model minus mean of the other models"
    else:
        kwargs = {"cmap": "viridis"}
    cbar_label = metric
    sns.heatmap(
        values,
        annot=_star_labels(values, q, alpha, True),
        fmt="",
        annot_kws={"size": 7},
        xticklabels=[g for _, g in keys],
        linewidths=0.5,
        linecolor="white",
        cbar_kws={"label": cbar_label, "shrink": 0.7},
        ax=ax,
        **kwargs,
    )

    # Dimension names above the columns, thick separators between dimensions
    start = 0
    for dim, grp in groupby(keys, key=lambda key: key[0]):
        width = len(list(grp))
        ax.text(start + width / 2, 1.01, dim, transform=ax.get_xaxis_transform(), ha="center", va="bottom", fontweight="bold")
        if start:
            ax.axvline(start, color="black", lw=2)
        start += width

    noun = "user groups" if side == "user" else "item groups"
    ax.set_title(
        f"Where is each model strong? {metric} per {noun}\n"
        + ("Blue = better than the average of the other models, red = worse. " if value == "lift" else "")
        + f"* = significant (FDR q < {alpha})",
        pad=30,
    )
    ax.set_xlabel("")
    ax.set_ylabel("")
    plt.setp(ax.get_xticklabels(), rotation=60, ha="right")

    out_path = Path(output_dir)
    out_path.mkdir(parents=True, exist_ok=True)
    filename = f"model_profile_{side}_{value}_{_sanitize_filename(metric)}.png"
    filepath = out_path / filename

    plt.savefig(filepath, bbox_inches="tight", dpi=300)
    plt.close(fig)  # Prevents memory accumulation in loops
    return filepath


def _sanitize_filename(name: str) -> str:
    """Helper to convert strings into clean, cross-platform filename paths."""
    clean = name.replace("@", "_at_").replace(" ", "_").lower()
    return re.sub(r"[^\w\-]", "", clean)


def plot_pair(
    cross: pd.DataFrame,
    winners: pd.DataFrame,
    user_dim: str,
    item_dim: str,
    user_groups: GroupMasks,
    item_groups: GroupMasks,
    models: list[str],
    metric: str = f"recall@{K}",
    alpha: float = ALPHA,
    max_annotated_cells: int = 6000,
    output_dir: str | Path = Path("..", "..", "figures"),
) -> Path:
    """Winner map + one lift panel per model for a (user dim x item dim) matrix."""
    cells = cross[(cross["user_dim"] == user_dim) & (cross["item_dim"] == item_dim)]
    won = winners[(winners["user_dim"] == user_dim) & (winners["item_dim"] == item_dim)]
    if cells.empty:
        raise ValueError(f"No cells for {user_dim} x {item_dim} (all below the size thresholds?).")

    rows = [g for g in user_groups[user_dim] if g in set(cells["user_group"])]
    cols = [g for g in item_groups[item_dim] if g in set(cells["item_group"])]
    row_labels = [f"{g} (n={int(user_groups[user_dim][g].sum())} users)" for g in rows]
    col_labels = [f"{g} ({int(item_groups[item_dim][g].sum())} items)" for g in cols]
    annotate_values = len(rows) * len(cols) <= max_annotated_cells  # big matrices: colours and stars only

    def grid(frame: pd.DataFrame, column: str) -> pd.DataFrame:
        return frame.pivot(index="user_group", columns="item_group", values=column).reindex(index=rows, columns=cols)

    limit = cells["lift"].abs().quantile(0.98) or 1.0
    n_panels = len(models) + 1
    ncols = min(5, n_panels)
    nrows = math.ceil(n_panels / ncols)

    # Dynamically scale column width based on the number of item groups
    panel_width = max(5.0, 0.35 * len(cols))
    panel_height = max(3.0, 0.4 * len(rows) + 2.5)

    # Dynamically shrink font size as the matrix grows (bounds between 4pt and 7pt)
    cell_fontsize = max(4, min(7, int(60 / max(len(rows), len(cols)))))
    fig, axes = plt.subplots(nrows, ncols, figsize=(panel_width * ncols, nrows * panel_height), constrained_layout=True, squeeze=False)
    flat = axes.ravel()
    for ax in flat[n_panels:]:
        ax.set_visible(False)

    # Panel 0: winner map
    model_idx = {m: i for i, m in enumerate(models)}
    best = grid(won, "best")
    codes = best.apply(lambda column: column.map(model_idx)).astype(float)
    decisive = grid(won, "margin_q") < alpha
    labels = pd.DataFrame(np.where(decisive, "*", ""), index=best.index, columns=best.columns)
    sns.heatmap(
        codes,
        cmap=ListedColormap([MODEL_COLORS[m] for m in models]),
        vmin=-0.5,
        vmax=len(models) - 0.5,
        annot=labels,
        fmt="",
        annot_kws={"size": cell_fontsize, "color": "white"},
        cbar=False,
        xticklabels=col_labels,
        yticklabels=row_labels,
        linewidths=0.5,
        linecolor="white",
        ax=flat[0],
    )
    flat[0].set_facecolor("lightgrey")
    flat[0].set_title("Best model per cell\n(* = significantly ahead of the runner-up)", fontsize=10)

    # Panels 1..n: each model relative to the others
    for j, (ax, model) in enumerate(zip(flat[1:n_panels], models, strict=True), start=1):
        frame = cells[cells["model"] == model]
        lift, q = grid(frame, "lift"), grid(frame, "lift_q")
        sns.heatmap(
            lift,
            cmap="RdBu",
            vmin=-limit,
            vmax=limit,
            annot=_star_labels(lift, q, alpha, annotate_values),
            fmt="",
            annot_kws={"size": cell_fontsize},
            cbar=False,
            xticklabels=col_labels,
            yticklabels=row_labels if j % ncols == 0 else False,
            linewidths=0.5,
            linecolor="white",
            ax=ax,
        )
        ax.set_facecolor("lightgrey")
        ax.set_title(f"{model}\nvs. mean of other models", fontsize=10, color=MODEL_COLORS[model], fontweight="bold")
    for ax in flat[:n_panels]:
        ax.set_xlabel(item_dim)
        ax.set_ylabel(user_dim)
        plt.setp(ax.get_xticklabels(), rotation=60, ha="right", fontsize=8)

    fig.colorbar(
        ScalarMappable(Normalize(-limit, limit), cmap="RdBu"),
        ax=list(flat[:n_panels]),
        location="bottom",
        shrink=0.4,
        label=f"{metric}: model minus mean of the other models (blue = better)",
    )
    fig.legend(
        handles=[Patch(color=MODEL_COLORS[m], label=m) for m in models if m in set(won["best"])], title="Best model", loc="outside right center"
    )
    fig.suptitle(
        f"{user_dim} × {item_dim}: {metric}\n"
        f"Rows = user groups, columns = item groups (cells = test items of that item group for users of that user group). "
        f"* = significant (FDR q < {alpha}). Grey = too few users or test items.",
        fontsize=11,
    )

    out_path = Path(output_dir)
    out_path.mkdir(parents=True, exist_ok=True)
    filename = f"pair_{_sanitize_filename(user_dim)}_x_{_sanitize_filename(item_dim)}_{_sanitize_filename(metric)}.png"
    filepath = out_path / filename

    plt.savefig(filepath, bbox_inches="tight", dpi=400)
    plt.close(fig)  # Prevents memory accumulation in loops
    return filepath

In [56]:
data = build_evaluation_data(df, MODELS)
user_groups, item_groups = make_user_groups(data), make_item_groups(data)

cells, winners = screen_cells(data, user_groups, item_groups, MODELS)
cross = add_interaction(cells)

# 1. Where is each model strong? (the headline plots for the task)
plot_model_profile(cells, MODELS, side="user")
plot_model_profile(cells, MODELS, side="item")

# # 2. Which dimension combinations deserve a closer look?
# pairs = rank_dimension_pairs(cross, winners)
# print(pairs.head(10))

# # 3. Plot the top combinations
# for user_dim, item_dim in pairs.head(3).index:
#     plot_pair(cross, winners, user_dim, item_dim, user_groups, item_groups, MODELS)

diverse_pairs, dominant_pairs = rank_dimension_pairs_v2(cross, winners)
print("--- TOP 3 COMPETITIVE PLOTS (Model Trade-offs) ---")
print(diverse_pairs[["n_distinct_winners", "share_interacting", "dominant_model"]].head(3))

print("\n--- TOP 3 DOMINANCE PLOTS (Clear Model Victories) ---")
print(dominant_pairs[["dominant_model", "dominant_share", "share_decisive", "mean_margin"]].head(3))

# Plot top competitive pair
for user_dim, item_dim in diverse_pairs.head(3).index:
    plot_pair(cross, winners, user_dim, item_dim, user_groups, item_groups, MODELS, output_dir=Path("..", "..", "figures", "diverse"))

# Plot top dominant pair
for user_dim_dom, item_dim_dom in dominant_pairs.head(3).index:
    plot_pair(cross, winners, user_dim_dom, item_dim_dom, user_groups, item_groups, MODELS, output_dir=Path("..", "..", "figures", "dominant"))

--- TOP 3 COMPETITIVE PLOTS (Model Trade-offs) ---
                     n_distinct_winners  share_interacting dominant_model
user_dim   item_dim                                                      
occupation genre                     16           0.002033    SLIMElastic
age_group  decade                    14           0.004167          NeuMF
activity   genre                     14           0.002119  TunedWeighted

--- TOP 3 DOMINANCE PLOTS (Clear Model Victories) ---
                           dominant_model  dominant_share  share_decisive  \
user_dim item_dim                                                           
gender   popularity_bucket       LightGCN        0.333333             0.0   
activity popularity_bucket       LightGCN        0.300000             0.0   
         genre              TunedWeighted        0.237288             0.0   

                            mean_margin  
user_dim item_dim                        
gender   popularity_bucket     0.003339  
activity pop

In [50]:
# 4. Concrete findings as tables
notable_cells(cross, by="lift")  # biggest significant wins/losses

,user_dim,user_group,item_dim,item_group,n_users,n_test,model,score,lift,se,p,lift_q,user_lift,item_lift,overall_lift,expected_lift,interaction,interaction_q
2843,occupation,administrator,decade,1970s,28,56,TunedCombination,0.562075,0.146995,0.039480,9.159930e-04,2.119095e-02,0.036105,0.027713,0.021595,0.042224,0.104772,0.254715
6355,activity,mid-high,decade,1970s,103,149,ItemKNN,0.250809,-0.122654,0.027536,2.160207e-05,1.172350e-03,-0.041650,-0.079868,-0.033470,-0.088048,-0.034606,1.000000
6627,activity,mid-high,genre,Mystery,91,108,ItemKNN,0.142857,-0.112210,0.032898,9.715956e-04,2.211963e-02,-0.041650,-0.050278,-0.033470,-0.058457,-0.053753,1.000000
2003,gender,F,decade,1980s,111,274,ItemKNN,0.360957,0.099950,0.023394,4.124642e-05,2.032462e-03,-0.031915,0.066142,-0.033470,0.067697,0.032254,1.000000
5602,occupation,writer,genre,Action,34,117,FISM,0.200972,-0.098427,0.029056,1.839061e-03,3.620426e-02,-0.004570,-0.053920,-0.020734,-0.037756,-0.060670,0.678391
163,age_group,18-25,decade,1970s,92,154,ItemKNN,0.269384,-0.095012,0.030113,2.174225e-03,4.052537e-02,-0.030595,-0.079868,-0.033470,-0.076994,-0.018018,1.000000
179,age_group,18-25,decade,1980s,114,292,ItemKNN,0.426671,0.094798,0.025091,2.537747e-04,8.300042e-03,-0.030595,0.066142,-0.033470,0.069017,0.025782,1.000000
1527,age_group,46-55,genre,Adventure,61,133,LightGCN,0.252186,-0.094499,0.029440,2.133544e-03,4.022682e-02,-0.005119,-0.043386,-0.002383,-0.046122,-0.048377,1.000000
1139,age_group,36-45,genre,Animation,46,71,ItemKNN,0.097826,-0.092150,0.028496,2.290704e-03,4.213184e-02,-0.037281,-0.046503,-0.033470,-0.050314,-0.041836,1.000000
2419,gender,M,decade,1970s,272,493,ItemKNN,0.279035,-0.088556,0.018184,1.898547e-06,1.773155e-04,-0.034103,-0.079868,-0.033470,-0.080502,-0.008054,1.000000


In [51]:
notable_cells(cross, by="interaction", model="EASE")  # surprising cells for one model

,user_dim,user_group,item_dim,item_group,n_users,n_test,model,score,lift,se,p,lift_q,user_lift,item_lift,overall_lift,expected_lift,interaction,interaction_q
4001,occupation,librarian,popularity_bucket,mid,35,89,EASE,0.000000,-0.002812,0.002546,0.277222,0.592825,0.056298,-0.002560,0.013231,0.040506,-0.043318,9.496675e-63
1761,age_group,56+,popularity_bucket,mid,25,83,EASE,0.000000,-0.005703,0.003957,0.162408,0.480344,0.033223,-0.002560,0.013231,0.017431,-0.023134,3.039806e-07
1473,age_group,46-55,popularity_bucket,mid,74,194,EASE,0.002252,-0.001818,0.001509,0.232355,0.553463,0.025094,-0.002560,0.013231,0.009303,-0.011120,1.206429e-11
6849,activity,high,popularity_bucket,mid,227,1135,EASE,0.000551,-0.002417,0.000694,0.000593,0.015130,0.006350,-0.002560,0.013231,-0.009441,0.007024,3.836960e-22
6833,activity,high,popularity_bucket,tail,93,192,EASE,0.000000,-0.000072,0.000072,0.319933,0.627804,0.006350,-0.000046,0.013231,-0.006926,0.006855,0.000000e+00
1073,age_group,36-45,popularity_bucket,mid,93,242,EASE,0.000000,-0.000800,0.000437,0.069937,0.321438,0.008816,-0.002560,0.013231,-0.006975,0.006175,2.522384e-43
2881,occupation,administrator,popularity_bucket,mid,42,109,EASE,0.000000,-0.000409,0.000298,0.177872,0.493612,0.010312,-0.002560,0.013231,-0.005480,0.005071,1.103760e-62
625,age_group,26-35,popularity_bucket,tail,40,65,EASE,0.000000,-0.000167,0.000167,0.323475,0.627804,0.015249,-0.000046,0.013231,0.001972,-0.002139,1.265404e-35
2465,gender,M,popularity_bucket,tail,103,170,EASE,0.000000,-0.000065,0.000065,0.319677,0.627804,0.011670,-0.000046,0.013231,-0.001607,0.001542,3.054626e-123


In [52]:
notable_cells(cross, by="interaction")

,user_dim,user_group,item_dim,item_group,n_users,n_test,model,score,lift,se,p,lift_q,user_lift,item_lift,overall_lift,expected_lift,interaction,interaction_q
4563,occupation,programmer,decade,1960s,25,31,ItemKNN,0.120000,0.040000,0.032603,0.231769,0.553463,-0.055586,-0.051629,-0.033470,-0.073745,0.113745,2.058120e-02
4261,occupation,other,decade,1970s,34,55,SLIMElastic,0.306373,-0.065163,0.028968,0.031271,0.221041,0.015200,0.026472,0.009079,0.032593,-0.097757,3.062009e-02
5033,occupation,student,decade,1980s,97,217,TunedAugmentation,0.440378,0.063368,0.020188,0.002254,0.041808,-0.027960,0.010603,-0.013003,-0.004354,0.067721,3.275928e-02
5288,occupation,student,genre,Mystery,63,86,NGCF,0.159259,-0.023316,0.020009,0.248381,0.570995,0.014682,0.033758,0.005930,0.042510,-0.065826,3.992938e-02
5373,occupation,student,genre,Western,28,37,TunedMixed,0.000000,-0.047619,0.014791,0.003333,0.055111,0.019509,0.011667,0.014851,0.016325,-0.063944,7.604314e-04
6601,activity,mid-high,genre,Horror,76,111,TunedAugmentation,0.148026,-0.064313,0.015998,0.000137,0.005198,-0.021973,0.002356,-0.013003,-0.006615,-0.057698,1.364000e-02
1764,age_group,56+,popularity_bucket,mid,25,83,UserKNN,0.000000,-0.005703,0.003957,0.162408,0.480344,0.033176,-0.005561,-0.017096,0.044710,-0.050413,3.943665e-35
5366,occupation,student,genre,Western,28,37,NeuMF,0.000000,-0.047619,0.014791,0.003333,0.055111,0.004501,-0.011190,-0.007777,0.001088,-0.048707,3.969832e-02
135,age_group,18-25,decade,1950s,47,70,LightGCN,0.015957,-0.017376,0.014328,0.231440,0.553463,-0.002854,0.030292,-0.002383,0.029821,-0.047197,3.969832e-02
4079,occupation,librarian,genre,Comedy,36,147,TunedWeighted,0.149802,-0.003889,0.012357,0.754858,0.907921,0.047607,0.016531,0.022598,0.041540,-0.045429,1.067035e-02


In [53]:
# Filter for main body plots
MAIN_BODY_MODELS = ["UserKNN", "EASE", "LightGCN", "SLIMElastic", "TunedCombination"]
cells_main = cells[cells["model"].isin(MAIN_BODY_MODELS)]

# Generate clean, compact main body figures
plot_model_profile(cells_main, MAIN_BODY_MODELS, side="user")
plot_model_profile(cells_main, MAIN_BODY_MODELS, side="item")

PosixPath('../../figures/model_profile_item_lift_recall_at_10.png')

In [ ]:
def load_metadata_and_ground_truth(
    dataloader_instance,
    user_file: Path = Path("..", "..", "data", "ml-100k", "ml-100k.user"),
    item_file: Path = Path("..", "..", "data", "ml-100k", "ml-100k.item"),
    test_split_file: str = "test.tsv",
) -> tuple[dict, dict, dict]:
    """
    Loads user_info, item_info, and constructs the ground_truth dictionary.
    """
    # Load raw dicts from your class methods
    item_info = dataloader_instance.load_item_info(item_file)
    user_info = dataloader_instance.load_user_info(user_file)

    # Build ground_truth dict: (user_id, item_id) -> 1.0 (binary relevance)
    ground_truth = {}
    try:
        pairs = dataloader_instance.load_split_data_file(test_split_file)
        for row in pairs:
            if len(row) >= 2:
                u, i = int(row[0]), int(row[1])
                val = float(row[2]) if len(row) >= 3 else 1.0
                ground_truth[(u, i)] = val
    except (FileNotFoundError, AssertionError) as e:
        print(f"Warning: Could not build ground truth from {test_split_file}: {e}")

    return user_info, item_info, ground_truth


def generate_beyond_accuracy_matrix_v2(
    model_names: list[str], loader_func: Callable[[str], tuple], item_info: dict[int, tuple[str, int, list[str]]], k: int = 10
) -> pd.DataFrame:
    """
    Computes beyond-accuracy metrics using load_evaluation_data's 6-tuple output.
    """
    all_metrics = {}
    distance_matrix = None
    first_item_order = None

    for model_name in model_names:
        filename = f"{model_name}.npz"

        # Unpack your exact 6-tuple signature
        _, item_id2idx, scores, train_mask, test_mask, validation_mask = loader_func(filename)

        num_items = scores.shape[1]

        # Extract raw item IDs in exact matrix column order
        items_in_order = np.empty(num_items, dtype=np.int64)
        for raw_id, idx in item_id2idx.items():
            items_in_order[int(idx)] = int(raw_id)

        # Build distance matrix ONCE using the first model's item ordering
        if distance_matrix is None:
            first_item_order = items_in_order.copy()

            # Map index order to genres
            internal_item_features = [item_info.get(int(iid), ("", 0, []))[2] for iid in items_in_order]

            distance_matrix = BeyondAccuracyMetrics.build_genre_distance_matrix(internal_item_features, num_items, jaccard_distance)
        else:
            # Ensure item column alignment is identical across models
            if not np.array_equal(items_in_order, first_item_order):
                raise ValueError(f"{model_name}: item index order differs from the first model.")

        # Train + Validation form the historical hidden items
        hidden = train_mask | validation_mask
        relevant = test_mask

        # Compute metrics using your existing BeyondAccuracyMetrics class
        metrics = BeyondAccuracyMetrics.calculate_all_metrics(scores=scores, hidden=hidden, relevant=relevant, distance_matrix=distance_matrix, k=k)
        all_metrics[model_name] = metrics

    return pd.DataFrame.from_dict(all_metrics, orient="index")

In [ ]:
def generate_report_table(
    cells: pd.DataFrame, beyond_acc_df: pd.DataFrame, selected_models: list[str], metric_col: str = "recall@10", alpha: float = 0.05
) -> pd.DataFrame:
    """
    Generates Table 1 for the 1-page report main body.

    Extracts overall accuracy, identifies top user/item subgroup strengths
    based on statistically significant lift (q < alpha), and merges
    beyond-accuracy metrics.
    """
    table_rows = []

    for model in selected_models:
        model_cells = cells[cells["model"] == model]

        # Overall Model Accuracy (user_dim == 'all' and item_dim == 'all')
        overall_row = model_cells[(model_cells["user_dim"] == "all") & (model_cells["item_dim"] == "all")]
        overall_score = overall_row[metric_col].values[0] if not overall_row.empty else np.nan

        # Top User Subgroup Strength (item_dim == 'all', q < alpha, max lift)
        user_marginals = model_cells[(model_cells["item_dim"] == "all") & (model_cells["user_dim"] != "all") & (model_cells["lift_q"] < alpha)]
        if not user_marginals.empty:
            best_u = user_marginals.loc[user_marginals["lift"].idxmax()]
            top_user_str = f"{best_u['user_group']} (+{best_u['lift']:.2f})"
        else:
            top_user_str = "None (Neutral)"

        # Top Item Subgroup Strength (user_dim == 'all', q < alpha, max lift)
        item_marginals = model_cells[(model_cells["user_dim"] == "all") & (model_cells["item_dim"] != "all") & (model_cells["lift_q"] < alpha)]
        if not item_marginals.empty:
            best_i = item_marginals.loc[item_marginals["lift"].idxmax()]
            top_item_str = f"{best_i['item_group']} (+{best_i['lift']:.2f})"
        else:
            top_item_str = "None (Neutral)"

        # Extract Beyond-Accuracy Metrics
        b_metrics = beyond_acc_df.loc[model] if model in beyond_acc_df.index else {}

        table_rows.append(
            {
                "Model": model,
                f"Overall {metric_col.capitalize()}": f"{overall_score:.3f}" if not np.isnan(overall_score) else "N/A",
                "Top User Niche (Lift)": top_user_str,
                "Top Item Niche (Lift)": top_item_str,
                "ILD@10": f"{b_metrics.get('ild@10', np.nan):.3f}",
                "Novelty@10 (bits)": f"{b_metrics.get('novelty@10', np.nan):.2f}",
                "Coverage@10": f"{b_metrics.get('coverage@10', np.nan) * 100:.1f}%",
            }
        )

    summary_df = pd.DataFrame(table_rows)
    return summary_df


# Assuming `loader` is an active RecommenderDataLoader instance
user_info, item_info, ground_truth = load_metadata_and_ground_truth(RecommenderDataLoader(base_dir=Path("..", "..", "results")))

beyond_acc_df = generate_beyond_accuracy_matrix_v2(model_names=MODELS, loader_func=load_evaluation_data, item_info=item_info, k=10)
display(cells.head(3))
display(beyond_acc_df.head(3))
table_1 = generate_report_table(cells=cells, beyond_acc_df=beyond_acc_df, selected_models=MAIN_BODY_MODELS, metric_col="score", alpha=0.05)

# Export to LaTeX for direct compilation into your report document
print(table_1.to_latex(index=False, caption="Model Subgroup Strengths and Beyond-Accuracy Metrics", label="tab:model_summary"))

,user_dim,user_group,item_dim,item_group,n_users,n_test,model,score,lift,se,p,lift_q
0,all,all,all,all,943,9596,BPR,0.268300,-0.000087,0.003481,0.980013,1.000000
1,all,all,all,all,943,9596,EASE,0.280786,0.013231,0.003550,0.000205,0.007019
2,all,all,all,all,943,9596,FISM,0.248944,-0.020734,0.004807,0.000018,0.001012


,coverage@10,novelty@10,ild@10,serendipity@10
BPR,0.304994,8.608081,0.801558,0.173253
EASE,0.232461,8.517709,0.809918,0.179110
FISM,0.372176,8.766175,0.804275,0.154901


\begin{table}
\caption{Model Subgroup Strengths and Beyond-Accuracy Metrics}
\label{tab:model_summary}
\begin{tabular}{lllllll}
\toprule
Model & Overall Score & Top User Niche (Lift) & Top Item Niche (Lift) & ILD@10 & Novelty@10 (bits) & Coverage@10 \\
\midrule
UserKNN & 0.252 & M (+-0.01) & Adventure (+0.04) & 0.811 & 8.42 & 17.4% \\
EASE & 0.281 & librarian (+0.06) & Thriller (+0.03) & 0.810 & 8.52 & 23.2% \\
LightGCN & 0.266 & None (Neutral) & mid (+0.01) & 0.786 & 8.75 & 35.9% \\
SLIMElastic & 0.277 & None (Neutral) & None (Neutral) & 0.812 & 8.53 & 23.5% \\
TunedCombination & 0.289 & administrator (+0.04) & Sci-Fi (+0.03) & 0.803 & 8.61 & 30.3% \\
\bottomrule
\end{tabular}
\end{table}



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


def plot_compact_combined_profile_flat(
    cells: pd.DataFrame,
    models: list[str],
    metric: str = "recall@10",
    alpha: float = 0.05,
    output_dir: str | Path = Path("..", "..", "figures"),
) -> Path:
    def get_pivot(side: str):
        dim_col, group_col, other_dim = ("user_dim", "user_group", "item_dim") if side == "user" else ("item_dim", "item_group", "user_dim")
        marginals = cells[(cells[other_dim] == "all") & (cells[dim_col] != "all")]
        keys = list(marginals[[dim_col, group_col]].drop_duplicates().itertuples(index=False, name=None))

        def pivot(column: str) -> pd.DataFrame:
            return marginals.pivot(index="model", columns=[dim_col, group_col], values=column).reindex(
                index=models, columns=pd.MultiIndex.from_tuples(keys)
            )

        return pivot("lift"), pivot("lift_q"), keys

    v_user, q_user, keys_user = get_pivot("user")
    v_item, q_item, keys_item = get_pivot("item")

    limit = (
        max(float(np.nanquantile(np.abs(v_user.to_numpy(dtype=float)), 0.98)), float(np.nanquantile(np.abs(v_item.to_numpy(dtype=float)), 0.98)))
        or 1.0
    )

    fig_width = 10.5
    fig_height = len(models) * 0.45 + 1.8
    fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(fig_width, fig_height), constrained_layout=True)

    def draw_heatmap(ax, values, q, keys, title):
        sns.heatmap(
            values,
            annot=False,
            cmap="RdBu",
            vmin=-limit,
            vmax=limit,
            cbar=False,
            ax=ax,
            xticklabels=[g for _, g in keys],
            linewidths=0.5,
            linecolor="white",
        )

        # Adding +0.08 to row_idx shifts the character slightly down in cell coordinates,
        # perfectly centering the elevated '*' glyph.
        y_offset = 0.25
        for row_idx in range(len(values)):
            for col_idx in range(len(values.columns)):
                q_val = q.iloc[row_idx, col_idx]
                if pd.notna(q_val) and q_val < alpha:
                    ax.text(col_idx + 0.5, row_idx + 0.5 + y_offset, "*", ha="center", va="center", fontsize=11, fontweight="bold", color="black")

        # Dimension headers and separators
        start = 0
        for dim, grp in groupby(keys, key=lambda key: key[0]):
            width = len(list(grp))
            ax.text(start + width / 2, 1.01, dim, transform=ax.get_xaxis_transform(), ha="center", va="bottom", fontweight="bold", fontsize=8)
            if start:
                ax.axvline(start, color="black", lw=1.5)
            start += width

        ax.set_title(title, pad=12, fontsize=9, fontweight="bold")
        ax.set_xlabel("")
        ax.set_ylabel("")

        plt.setp(ax.get_xticklabels(), rotation=45, ha="right", fontsize=7)
        plt.setp(ax.get_yticklabels(), fontsize=8)

    draw_heatmap(axes[0], v_user, q_user, keys_user, "User Group Profiles")
    draw_heatmap(axes[1], v_item, q_item, keys_item, "Item Group Profiles")

    fig.colorbar(
        ScalarMappable(Normalize(-limit, limit), cmap="RdBu"),
        ax=axes.ravel().tolist(),
        location="bottom",
        shrink=0.4,
        aspect=40,
        pad=0.01,
        label=f"{metric} Lift (Blue = Better)",
    )

    out_path = Path(output_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    def _sanitize_filename(name: str) -> str:
        clean = name.replace("@", "_at_").replace(" ", "_").lower()
        return re.sub(r"[^\w\-]", "", clean)

    filename = f"compact_combined_profile_flat_{_sanitize_filename(metric)}.png"
    filepath = out_path / filename

    plt.savefig(filepath, bbox_inches="tight", dpi=500)
    plt.close(fig)
    return filepath


# Generate the compact combined plot (returns path to output)
compact_plot_path = plot_compact_combined_profile_flat(
    cells=cells,  # Your main cells DataFrame
    models=MAIN_BODY_MODELS,
    metric="recall@10",
    alpha=0.05,
)

print(f"Generated compact matrix for report: {compact_plot_path}")

Generated compact matrix for report: ../../figures/compact_combined_profile_flat_recall_at_10.png
